In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import celltypist
from celltypist import models
from celltypist.models import Model

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_scrna03.h5ad")

In [ ]:
adata_scrna

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='celltypist_liver',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.tl.leiden(adata_scrna, resolution=0.1, key_added="11cluster")
sc.pl.umap(adata_scrna, color=['11cluster'])

In [ ]:
adata_scrna

In [ ]:
sc_pre = sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/pre/MASLD_scrna01.h5ad")
sc_pre

In [ ]:
sc_pre.obs['n_counts'] = sc_pre.X.sum(axis=1).A1
sc_pre.obs['n_genes'] = (sc_pre.X > 0).sum(axis=1).A1
sc_pre.obs['mt_percent'] = adata_scrna.obs['mt_percent']
sc_pre.obs['sample'] = adata_scrna.obs['sample']
sc_pre.obs['state'] = adata_scrna.obs['state']
sc_pre.obs['11cluster'] = adata_scrna.obs['11cluster']
sc_pre.obs['celltypist_liver'] = adata_scrna.obs['celltypist_liver']
sc_pre

In [ ]:
sc_mp = sc_pre[sc_pre.obs['11cluster'].isin(['1'])].copy()
sc_mp

In [ ]:
sc_mp.obs['state'].value_counts()

In [ ]:
sc_mp.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/pre/MASLD_scrna_MP01.h5ad")

In [ ]:
sc.pp.normalize_total(sc_mp, target_sum=1e4)
sc.pp.log1p(sc_mp)
sc.pp.highly_variable_genes(sc_mp, flavor='seurat', n_top_genes=2000)

In [ ]:
# Scale data
sc.pp.scale(sc_mp, max_value=10)
# Run PCA
sc.tl.pca(sc_mp, svd_solver='arpack')
print(sc_mp.obsm['X_pca'][:5, :5])
sc.pl.pca(sc_mp, color='sample')

import matplotlib.pyplot as plt
var_ratio = sc_mp.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(sc_mp, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(sc_mp, n_pcs=30, use_rep='X_pca')
sc.tl.umap(sc_mp)
sc.pl.umap(sc_mp, color=['sample'])

In [ ]:
sc.pl.umap(sc_mp, color=['state'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(sc_mp.obsm['X_pca'],sc_mp.obs, 'sample')

# Harmony 결과를 저장
sc_mp.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(sc_mp, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(sc_mp)

In [ ]:
sc_mp

In [ ]:
sc.pl.umap(sc_mp,color='state')

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 차원 축소된 데이터 (예: PCA or UMAP 등)
X = sc_mp.obsm['X_pca']  # 또는 'X_umap' 등

inertias = []
k_range = range(2, 30)  

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)  # inertia = 클러스터 내부 거리 제곱합

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

In [ ]:
sc_mp.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_sc_MP03.h5ad")

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_scrna03.h5ad")

In [ ]:
adata_scrna

In [ ]:
sc_mp.obs['cellannot_detail_JHB'] = adata_scrna.obs['cellannot_detail_JHB']

In [ ]:
sc.tl.leiden(sc_mp, resolution=0.5, key_added="10cluster")
sc.pl.umap(sc_mp, color=['10cluster'])

In [ ]:
sc.pl.umap(sc_mp, color=['cellannot_detail_JHB'])

In [ ]:
sc_mp

In [ ]:
cluster_labels = sc_mp.obs['10cluster']
batch_labels = sc_mp.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

crosstab.plot(kind='bar', stacked=True, figsize=(5, 7))
plt.title('T cells Proportion per state')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='T subcelltype', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = sc_mp.obs['state']
cluster = sc_mp.obs['10cluster']

# 클러스터 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in clusters:  # 클러스터 번호 순 정렬
    # 각 조건별 개수 계산
    donor_in = ((state == "healthy") & (cluster == clus)).sum()
    donor_out = ((state == "healthy") & (cluster != clus)).sum()
    masld_in = ((state == "NAFLD") & (cluster == clus)).sum()
    masld_out = ((state == "NAFLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [masld_in, masld_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out)
    masld_frac = masld_in / (masld_in + masld_out)
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "masld_fraction": masld_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 정렬
df_result = df_result.sort_values("p_value")
print(df_result)

In [ ]:
# cluster 열을 숫자형으로 바꾼 뒤 정렬
df_result['cluster'] = df_result['cluster']
df_result = df_result.sort_values('cluster')

import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='Donor')
plt.bar(x + width/2, df_result['masld_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('MP cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['masld_fraction'][i])
        plt.text(x[i], 0.348, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
sc_mp_pre = sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/pre/MASLD_scrna_MP01.h5ad")
sc_mp_pre

In [ ]:
sc_mp_pre.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_sc_MP01.h5ad")

In [ ]:
sc_mp_pre=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_sc_MP01.h5ad")

In [ ]:
sc.pp.normalize_total(sc_mp_pre, target_sum=1e4)
sc.pp.log1p(sc_mp_pre)

In [ ]:
sc_mp_pre.obs_names_make_unique()

In [ ]:
# Healthy_Human_Liver.pkl
predictions = celltypist.annotate(sc_mp_pre, model = 'Immune_All_Low.pkl', majority_voting = True)

sc_mp_pre = predictions.to_adata()
sc_mp.obs['celltypist_Immune']= sc_mp_pre.obs['majority_voting']

In [ ]:
sc.pl.umap(sc_mp, color=['celltypist_Immune'])

In [ ]:
sc.pl.umap(sc_mp, color=['10cluster'])

In [ ]:
sc_mp.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_sc_MP04.h5ad")

In [ ]:
sc.pl.umap(sc_mp, color=['SLC7A5'])

In [ ]:
sc.pl.umap(sc_mp, color=['SLC7A5','SLC7A11','SLC3A2','SLC7A2'])

In [ ]:
sc.pl.umap(sc_mp, color=['CD14','CD1A','CD209'])

In [ ]:
sc.pl.umap(sc_mp, color=['VSIG4','FOLR2','CD163'])

In [ ]:
sc.pl.dotplot

In [ ]:
sc.pl.dotplot(sc_mp, ['CD14','CD1A','CD209','SLC7A5', 'SLC7A11', 'SLC3A2', 'SLC7A2'], groupby = 'state')

In [ ]:
KC_marker = ['VSIG4','CLEC4F','FOLR2','CD5L','SLC40A1','MARCO','CD163'] # Martin.G and Charlotte.L.S (2022)
moKC_marker = [ 'VSIG4','FOLR2','CLEC4F','CLEC1B','CD5L','MARCO','GFRA2','ADRB1','TMEM26','SLC40A1','HMOX1','SLC16A9','VCAM1','SUCNR1','CD207']
LCM_marker = ['ADGRE1','FCGR1A','CD207','CX3CR1','CCR2']

In [ ]:
sc.pl.dotplot(sc_mp, KC_marker, groupby = '10cluster', standard_scale="var")

In [ ]:
sc.pl.dotplot(sc_mp, moKC_marker, groupby = '10cluster', standard_scale=None)

In [ ]:
sc.pl.dotplot(sc_mp, LCM_marker, groupby = '10cluster', standard_scale=None)

In [ ]:
import matplotlib.pyplot as plt

sc.pl.violin(
    sc_mp,
    keys=KC_marker,
    groupby="10cluster",
    multi_panel=True,  # ✅ 유전자별로 쌓여서 나옴
    rotation=45,
    show=True
)

In [ ]:
sc_mp

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 2))

sc.pl.violin(
    sc_mp,
    keys="SLC7A5",
    groupby="10cluster",
    multi_panel=False,
    show=True,
    ax=ax  # 만들어 둔 ax에 그림을 그림
)

In [ ]:
transporter_genes = ['SLC7A5', 'SLC7A11', 'SLC3A2', 'SLC7A2']

sc.pl.violin(
    sc_mp,
    keys=transporter_genes,
    groupby="state",
    multi_panel=True,  # 유전자마다 subplot 하나씩
    show=True
)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))

sc.pl.violin(
    sc_mp,
    keys="SLC7A5",
    groupby="celltypist_Immune",
    multi_panel=False,
    show=False,  # ← 여기서 False로 해야 아래에서 수동 조정 가능
    ax=ax
)

# X축 레이블 회전
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
sc.tl.dendrogram(sc_mp,groupby='10cluster')
sc.tl.rank_genes_groups(sc_mp, groupby='10cluster', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(sc_mp, n_genes=20)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(sc_mp, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    sc_mp,
    var_names=top_genes,
    groupby="10cluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
sc.tl.dendrogram(sc_mp,groupby='10cluster')
sc.tl.rank_genes_groups(sc_mp, groupby='10cluster',use_raw=True, method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(sc_mp, n_genes=20)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(sc_mp, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    sc_mp,
    var_names=top_genes,
    groupby="10cluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
rank_df

In [ ]:
# 예시: 찾고 싶은 유전자 리스트
genes_of_interest = ["SLC7A5"]

# rank_df에서 필터링
found_genes = rank_df[rank_df['names'].isin(genes_of_interest)]
print(found_genes)

In [ ]:
sc.pl.rank_genes_groups_tracksplot(sc_mp, n_genes=3)

In [ ]:
sc.tl.rank_genes_groups(
    sc_mp,
    groupby="state",        # 비교할 obs 열
    groups=["NAFLD"],     # 비교할 그룹
    reference="healthy",     # 참조 그룹
    method="wilcoxon"       # 또는 't-test', 'logreg'
)

In [ ]:
sc_mp_pre=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_sc_MP01.h5ad")
sc.pp.normalize_total(sc_mp_pre, target_sum=1e4)
sc.pp.log1p(sc_mp_pre)

In [ ]:
sc_mp.raw = sc_mp_pre

In [ ]:
sc.tl.rank_genes_groups(
    sc_mp,
    groupby="state",        # 비교할 obs 열
    groups=["NAFLD"],     # 비교할 그룹
    reference="healthy",     # 참조 그룹
    use_raw=True,
    method="wilcoxon"       # 또는 't-test', 'logreg'
)

In [ ]:
deg_df = sc.get.rank_genes_groups_df(sc_mp, group="NAFLD")
print(deg_df.head())

In [ ]:
deg_df = sc.get.rank_genes_groups_df(sc_mp, group="NAFLD")
print(deg_df.head())

In [ ]:
deg_df['-log10(padj)'] = -np.log10(deg_df['pvals_adj'])
genes_of_interest = ["SLC7A5"]

plt.figure(figsize=(7, 6))
plt.scatter(
    deg_df['logfoldchanges'], 
    deg_df['-log10(padj)'], 
    c="grey", s=10, alpha=0.7
)

# 유의미한 cutoff 설정 (예: logFC > 1, padj < 0.05)
sig_up = (deg_df['logfoldchanges'] > 1) & (deg_df['pvals_adj'] < 0.05)
sig_down = (deg_df['logfoldchanges'] < -1) & (deg_df['pvals_adj'] < 0.05)

plt.scatter(deg_df.loc[sig_up, 'logfoldchanges'],
            deg_df.loc[sig_up, '-log10(padj)'],
            c="red", s=10, label="Upregulated")

plt.scatter(deg_df.loc[sig_down, 'logfoldchanges'],
            deg_df.loc[sig_down, '-log10(padj)'],
            c="blue", s=10, label="Downregulated")

# cutoff 선
plt.axvline(1, color="black", linestyle="--")
plt.axvline(-1, color="black", linestyle="--")
plt.axhline(-np.log10(0.05), color="black", linestyle="--")

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 Adjusted p-value")
plt.title("Volcano Plot: Disease vs Normal")
plt.legend()

for gene in genes_of_interest:
    if gene in deg_df['names'].values:
        x = deg_df.loc[deg_df['names'] == gene, 'logfoldchanges'].values[0]
        y = deg_df.loc[deg_df['names'] == gene, '-log10(padj)'].values[0]
        plt.text(x, y, gene, fontsize=8, color="black")  # 텍스트 표시


plt.tight_layout()
plt.show()

In [ ]:
deg_df